# 🏦 Demo 01 — Load Bank Accounts into Fabric Lakehouse

**What we're doing:** Loading a CSV file of bank accounts into a Delta Table and querying it.

> Run each cell by pressing the ▶ button or **Shift + Enter**

## Cell 1 — Load the CSV file from Lakehouse Files

In [ ]:
# Read the uploaded bank accounts CSV from the Lakehouse Files area into a Spark DataFrame
df = spark.read.option('header', 'true').option('inferSchema', 'true') \
    .csv('Files/sample_accounts.csv')

# Count the loaded rows and preview the first 5 records to confirm the file loaded correctly
print(f'✅ Loaded {df.count()} bank accounts')
df.show(5)

## Cell 2 — Save as a Delta Table (permanent, queryable storage)

In [ ]:
# Write the DataFrame to a managed Delta table in the Lakehouse, replacing any earlier copy
df.write.format('delta').mode('overwrite').saveAsTable('bank_accounts')

# Confirm that the Delta table is now available for downstream SQL queries
print('✅ Delta Table "bank_accounts" created successfully!')

## Cell 3 — Show all accounts

In [ ]:
%%sql
-- Return the main bank account columns from the Delta table for review
-- Sort the results by balance so the largest accounts appear first
SELECT AccountID, CustomerName, AccountType, Balance, Branch, Status
FROM bank_accounts
ORDER BY Balance DESC

## Cell 4 — Find high-value active savings accounts

In [ ]:
%%sql
-- Return only savings accounts that are active and hold more than $10,000
-- Order the matching customers from the highest balance to the lowest
SELECT CustomerName, AccountType, Balance, Branch
FROM bank_accounts
WHERE AccountType = 'Savings'
  AND Balance > 10000
  AND Status = 'Active'
ORDER BY Balance DESC

## Cell 5 — Summary statistics by branch

In [ ]:
%%sql
-- Aggregate active accounts by branch to compare account volume and balances
-- Sort branches by total balance so the highest-value branch appears first
SELECT 
    Branch,
    COUNT(*) AS TotalAccounts,
    ROUND(SUM(Balance), 2) AS TotalBalance,
    ROUND(AVG(Balance), 2) AS AvgBalance
FROM bank_accounts
WHERE Status = 'Active'
GROUP BY Branch
ORDER BY TotalBalance DESC